In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from cmdstanpy import CmdStanModel
import arviz as az
from sklearn.mixture import GaussianMixture

from lib.models.utils import (
    PerceptualModule,
    ResponseModule,
    get_stan_model_paths
)
from lib import filepaths

/mnt/c/Users/marco/Desktop/research/ibl_gc/.venv/lib/python3.13/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [2]:
dataset = pd.read_csv(filepaths.ROOT_BEHAV_DATA)
subj_ids_high_sessions = (
    dataset.groupby("subj_id_code")
    .session_id_code.nunique()
    .reset_index()
    .query("session_id_code >= 4")
    .subj_id_code.values
)

In [146]:
def get_posterior_summary(fit, var_names):
    idata = az.from_cmdstanpy(posterior=fit)
    summ = az.summary(idata, var_names=var_names, round_to=None)
    return summ

def ebfmi_one_chain(energy):
    energy = np.asarray(energy, dtype=float)
    return np.mean(np.diff(energy) ** 2) / np.var(energy, ddof=1)

def get_hmc_diagnostics(fit, max_treedepth):
    mv = fit.method_variables()

    div = np.squeeze(mv["divergent__"])   # draws x chains
    td  = np.squeeze(mv["treedepth__"])   # draws x chains
    en  = np.squeeze(mv["energy__"])      # draws x chains

    if div.ndim == 1:
        div = div[:, None]
        td = td[:, None]
        en = en[:, None]

    ebfmi = [ebfmi_one_chain(en[:, c]) for c in range(en.shape[1])]

    return {
        "n_divergent": int(div.sum()),
        "n_max_treedepth": int((td >= max_treedepth).sum()),
        "ebfmi_per_chain": ebfmi,
        "min_ebfmi": float(np.min(ebfmi)),
    }
    
def check_bimodality(x):
    x = x[:,None]
    gm1 = GaussianMixture(1).fit(x)
    gm2 = GaussianMixture(2).fit(x)
    bic1 = gm1.bic(x)
    bic2 = gm2.bic(x)
    return bic1, bic2

###

def posterior_test(
    fit,
    pars: list[str],
    mcse_less_than=0.05,
    divergence_less_then=0,
    treedepth_less_then=0,
    ebfmi_greater_then=0.3,
    rhat_less_then=1.01,
    bulk_greater_then=1000
):
    hmc = get_hmc_diagnostics(fit, max_treedepth=12)
    summary = get_posterior_summary(fit, pars)
    n_samples = fit.stan_variable(pars[0]).size
    
    mcse = summary["mcse_mean"]/summary["sd"]
    test1 = mcse.mean() <= mcse_less_than

    test2 = (
        (hmc["n_divergent"]/n_samples) <= divergence_less_then
        and (hmc["n_max_treedepth"]/n_samples) <= treedepth_less_then
        and hmc["min_ebfmi"] > ebfmi_greater_then
    )

    test3 = (
        np.all(summary["r_hat"] <= rhat_less_then)
        and summary.ess_bulk.mean() >= bulk_greater_then
        and summary.ess_bulk.iloc[0] >= bulk_greater_then
        and summary.ess_tail.mean() >= bulk_greater_then
        and summary.ess_tail.iloc[0] >= bulk_greater_then
    )

    non_bimodal = [np.diff(check_bimodality(fit.stan_variable(v)))[0] < 0 for v in pars]
    test4 = np.all(non_bimodal)
    
    return np.all([test1, test2, test3, test4])

In [52]:
parameters_map = dict(
    single_session=dict(
        scpw_basic=[],
        scpw_brt=[
            "omega2", 
            "beta_prior", "beta_sens", 
            "contrast_slope", "contrast_midpoint",
            "lapse"
        ],
        scpw_brrt=[]
    ),
    single_subject=dict(
        scpw_basic=[],
        scpw_brt=[
            "omega2_sess", 
            "beta_prior_sess", "beta_sens_sess", 
            "contrast_slope", "contrast_midpoint",
            "lapse"
        ],
        scpw_brrt=[]
    ),
    full=dict(
        scpw_basic=[],
        scpw_brt=[
            "omega2_session", 
            "beta_prior_session", "beta_sens_session", 
            "contrast_slope_subj", "contrast_midpoint_subj",
            "lapse_subj"
        ],
        scpw_brrt=[]
    )
)

In [53]:
hierarchy = "single_session"

fit_root = Path(filepaths.ROOT_STAN_MODEL_FITS)/hierarchy/"2level"
fnames = os.listdir(fit_root)

df = []
for f in fnames:
    perceptual_model = [s for s in f.split("-") if "perc__" in s][0].split("__")[-1]
    response_model = [s for s in f.split("-") if "res__" in s][0].split("__")[-1][:-4]
    df.append(dict(
        eid=f.split("-perc")[0],
        perceptual_model=perceptual_model,
        response_model=response_model,
        parameter_names=parameters_map[hierarchy][f"{perceptual_model}_{response_model}"],
        filepath=fit_root/f
    ))
    
df = pd.DataFrame(df)

In [57]:
for i, row in df.iterrows():
    fit = np.load(row.filepath, allow_pickle=True).item()
    pars = row.parameter_names
    break